# Quran Journey — character voices with Habibi-TTS (free, Google Colab GPU)
1. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
2. Wait ~15 min. At the end a file `voices.zip` downloads.
3. Unzip it into the project folder `tools/voice_raw/` and tell Claude — it converts them into the game.

Models: Habibi-TTS *Specialized* EGY + MSA (Apache-2.0). Reference voices: the Habibi Space examples (placeholders).

In [ ]:
# Install Habibi WITHOUT re-downloading PyTorch (Colab already has a CUDA torch; re-installing it
# pulls ~1.6 GB and can break the connection). The pins keep Colab's own torch/torchaudio.
import torch, torchaudio, importlib.util
v = lambda m: m.__version__.split('+')[0]
pins = f"torch=={v(torch)}\ntorchaudio=={v(torchaudio)}\n"
if importlib.util.find_spec('torchvision'):
    import torchvision; pins += f"torchvision=={v(torchvision)}\n"
open('pins.txt', 'w').write(pins); print(pins)
!pip -q install --retries 10 --timeout 120 -c pins.txt f5-tts habibi-tts
# torchaudio >= 2.9 reads audio through torchcodec; older versions don't need it
if tuple(int(x) for x in v(torchaudio).split('.')[:2]) >= (2, 9):
    !pip -q install --retries 10 --timeout 120 -c pins.txt torchcodec
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — switch the runtime to T4 GPU!')


In [ ]:
import json, os, urllib.request
REPO = 'https://raw.githubusercontent.com/mohamedjs/3d-quranic/main/'
try:
    lines = json.load(urllib.request.urlopen(REPO + 'tools/voice_lines.json'))
    voices = json.load(urllib.request.urlopen(REPO + 'tools/voices.json'))
except Exception as e:
    print('GitHub fetch failed (', e, ') — upload tools/voice_lines.json and tools/voices.json:')
    from google.colab import files
    up = files.upload()
    lines = json.loads(up['voice_lines.json']); voices = json.loads(up['voices.json'])
print(len(lines), 'lines;', {k: v['ref'] for k, v in voices.items() if not k.startswith('_')})
ASSETS = 'https://huggingface.co/spaces/chenxie95/Habibi-TTS/resolve/main/assets/'
REFTXT = {'EGY.mp3': 'ايه الكلام. بقولك ايه. استخدم صوتي في المحادثات. استخدمه هيعجبك اوي.',
          'MSA.mp3': 'كان اللعيب حاضرًا في العديد من الأنشطة والفعاليات المرتبطة بكأس العالم، مما سمح للجماهير بالتفاعل معه والتقاط الصور التذكارية.',
          'MAR.mp3': 'إذا بغيتي شي صوت باللهجة المغربية للإعلانات ديالك هذا أحسن واحد غادي تلقاه.',
          'IRQ.wav': 'يعني ااا ما نقدر ناخذ وقت أكثر، ااا لأنه شروط كلش يحتاجلها وقت.',
          'ALG.wav': 'أنيا هكا باغية ناكل هكا أني ن نشوف فيها الحاجة هذيكا.'}
os.makedirs('refs', exist_ok=True)
for f in REFTXT: urllib.request.urlretrieve(ASSETS + f, 'refs/' + f)

In [ ]:
from cached_path import cached_path
from f5_tts.infer.utils_infer import load_model, load_vocoder, preprocess_ref_audio_text
from f5_tts.model import DiT
from habibi_tts.infer.utils_infer import infer_process
cfg = dict(dim=1024, depth=22, heads=16, ff_mult=2, text_dim=512, conv_layers=4)
vocoder = load_vocoder()
MODELS = {}
for tag, ck in (('EGY', 'Specialized/EGY/model_100000.safetensors'), ('MSA', 'Specialized/MSA/model_200000.safetensors')):
    MODELS[tag] = load_model(DiT, cfg, str(cached_path('hf://SWivid/Habibi-TTS/' + ck)),
                             vocab_file=str(cached_path('hf://SWivid/Habibi-TTS/Specialized/%s/vocab.txt' % tag)))
print('models ready')

In [ ]:
import soundfile as sf, torch
os.makedirs('voices', exist_ok=True)
REF = {}
for i, L in enumerate(lines):
    out = 'voices/' + L['id'].replace('/', '__') + '.wav'
    if os.path.exists(out): continue
    v = voices[L['speaker']]; ref = v['ref']
    if ref not in REF: REF[ref] = preprocess_ref_audio_text('refs/' + ref, REFTXT[ref], show_info=print)
    text = L['text'].replace('ﷺ', 'صلى الله عليه وسلم')
    torch.manual_seed(7)
    wav, sr, _ = infer_process(REF[ref][0], REF[ref][1], text, MODELS[v['lang'][:3]], vocoder, nfe_step=32, speed=1, show_info=lambda *a, **k: None)
    sf.write(out, wav, sr)
    print(f'{i + 1}/{len(lines)}', L['id'], round(len(wav) / sr, 1), 's')

In [ ]:
# convert to small mp3 before zipping (~5 MB instead of ~60 MB of WAV)
!mkdir -p voices_mp3 && for f in voices/*.wav; do ffmpeg -loglevel error -y -i "$f" -ac 1 -ar 24000 -b:a 64k "voices_mp3/$(basename "${f%.wav}").mp3"; done
!cd voices_mp3 && zip -q -r ../voices.zip . && cd .. && ls -la voices.zip
from google.colab import files; files.download('voices.zip')